# Cortex MLP Router GRPO training (Workstream B Phase 4 — SKELETON)

**Status:** scaffold with explicit `TODO(conv-point)` markers at every Cortex-dependent integration site. **Do not run end-to-end yet** — the cells marked TODO will fail until Workstream A's Session 13 ships:

- `cortex.metacognition.MetacognitionState` — the 24-dim source vector.
- `cortex.routing_policy.RoutingPolicy` — the trainable interface (Phase A §6).
- `cortex.council.Council` — the deliberation orchestrator that drives rollouts.
- `baselines.cortex_fixed_router.B3CortexFixedRouter` — generates the deterministic-router trajectories used as the training-data corpus.

**At convergence point** (when Session 13 lands), uncomment the TODO blocks, run `Runtime → Run all`, and the notebook trains the MLP for 200–500 steps.

**Architecture (locked by Phase A docs/CORTEX_ARCHITECTURE.md):**
- Decision 40: `MetacognitionState` → `np.ndarray[float, (24,)]`.
- Decision 39: small MLP head, discrete output over top-50 most-common `(kind, brain, subagent)` tuples.
- Decision 41: argmax (eval) / sampled (training) over the action distribution.

**Reward source:** Phase-1-fixed `outer_reward` ∈ [-1, 1] composed with the token-budget penalty via `training.reward_shaping.shape_reward`.

## 1. Install dependencies

Pure PyTorch MLP — no LLM dependencies (no Unsloth, no vLLM). Cortex's brains DO use LLMs, but the trainable router only sees the featurised `MetacognitionState`.

In [ ]:
%%capture
!pip install --upgrade pip
!pip install torch numpy pydantic huggingface_hub matplotlib openenv

## 2. Authenticate with Hugging Face

In [ ]:
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception:
    from huggingface_hub import login
    login()
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

assert HF_TOKEN, "HF_TOKEN required"

## 3. Clone CrisisWorldCortex (post-Cortex-Session-13 deploy)

**TODO(conv-point):** the target Space repo at convergence point will have `cortex/*` populated. Until then, this cell pulls the pre-Cortex repo and the Cortex-dependent imports below will fail.

In [ ]:
%%capture
!rm -rf /content/CrisisWorldCortex
!git clone https://huggingface.co/spaces/Angshuman28/CrisisWorldCortex /content/CrisisWorldCortex
%cd /content/CrisisWorldCortex
!pip install -e .

In [ ]:
import sys
sys.path.insert(0, "/content/CrisisWorldCortex")

from training.rollout_buffer import RolloutBuffer, TrajectoryStep
from training.reward_shaping import shape_reward, compose_episode_return
from training.eval_metrics import collapse_rate

# TODO(conv-point): uncomment when Cortex Session 13 lands.
# from cortex.schemas import MetacognitionState, RoutingAction, RouterStep
# from cortex.routing_policy import RoutingPolicy
# from cortex.council import Council
# from baselines.cortex_fixed_router import B3CortexFixedRouter
print("Phase-2 training utilities OK; Cortex imports gated until Session 13")

## 4. MLP architecture (Phase A Decision 39 + 40)

24-dim featurised MetacognitionState → 2 hidden layers (64 → 64) → 50 action logits.

Action vocabulary `ACTION_VOCAB` is the top-50 most-common `(kind, brain, subagent)` tuples observed in B3 deterministic-router trajectories — populated empirically at convergence point from collected B3 RouterSteps.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

FEATURE_DIM = 24  # Phase A Decision 40
ACTION_VOCAB_SIZE = 50  # Phase A Decision 39
HIDDEN_DIM = 64

class CortexRouterMLP(nn.Module):
    """24-dim MetacognitionState -> 50-way action logits.

    The exact action mapped to each output index is determined by
    ACTION_VOCAB (frozen post-data-collection, see cell 6).
    """

    def __init__(self, feature_dim: int = FEATURE_DIM, vocab_size: int = ACTION_VOCAB_SIZE,
                 hidden: int = HIDDEN_DIM):
        super().__init__()
        self.fc1 = nn.Linear(feature_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.head = nn.Linear(hidden, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return self.head(h)  # logits, shape (..., vocab_size)

    def sample_action(self, x: torch.Tensor, temperature: float = 1.0):
        """Sample an action index + return its log-prob (training rollouts)."""
        logits = self.forward(x) / max(temperature, 1e-6)
        dist = torch.distributions.Categorical(logits=logits)
        idx = dist.sample()
        return idx, dist.log_prob(idx)

    def argmax_action(self, x: torch.Tensor) -> int:
        """Eval-mode action choice (Phase A Decision 41)."""
        return int(self.forward(x).argmax(dim=-1))

policy = CortexRouterMLP()
print(policy)

## 5. Featurization: MetacognitionState -> 24-dim vector

**TODO(conv-point):** the schema below references `cortex.schemas.MetacognitionState` (Session 13).
Per Phase A `cortex/schemas.py` and `docs/CORTEX_ARCHITECTURE.md` §6, the featurization concatenates:
  - tick (1), round (1), one-hot phase (4) -> 6 dims
  - inter_brain_agreement (1), average_confidence (1), average_evidence_support (1),
    novelty_yield_last_round (1), collapse_suspicion (1), budget_remaining_frac (1),
    urgency (1), preserved_dissent_count (1), challenge_used_this_tick (1) -> 9 dims
  - 9 reserved (0.0-padded) for forward-compat extensions -> 9 dims

Total: 6 + 9 + 9 = 24.

In [ ]:
import numpy as np

PHASE_ORDER = ("divergence", "challenge", "narrowing", "convergence")

def featurize_metacognition_state(state) -> np.ndarray:
    """Convert MetacognitionState -> 24-dim float32 array.

    TODO(conv-point): change ``state`` typing to MetacognitionState
    (cortex.schemas) once Session 13 lands. The protocol below assumes
    duck-typed access to the 11 documented fields + the phase string.
    """
    out = np.zeros(FEATURE_DIM, dtype=np.float32)
    # First 6: tick, round, phase one-hot.
    out[0] = float(getattr(state, "tick", 0))
    out[1] = float(getattr(state, "round", 1))
    phase = getattr(state, "phase", "divergence")
    if phase in PHASE_ORDER:
        out[2 + PHASE_ORDER.index(phase)] = 1.0
    # Next 9: scalar metacog signals.
    out[6] = float(getattr(state, "inter_brain_agreement", 0.0))
    out[7] = float(getattr(state, "average_confidence", 0.0))
    out[8] = float(getattr(state, "average_evidence_support", 0.0))
    out[9] = float(getattr(state, "novelty_yield_last_round", 0.0))
    out[10] = float(getattr(state, "collapse_suspicion", 0.0))
    out[11] = float(getattr(state, "budget_remaining_frac", 1.0))
    out[12] = float(getattr(state, "urgency", 0.0))
    out[13] = float(getattr(state, "preserved_dissent_count", 0))
    out[14] = float(getattr(state, "challenge_used_this_tick", 0))
    # 15..23 reserved (0.0-padded for forward-compat extensions).
    return out

## 6. Action vocabulary (top-50 from B3 deterministic-router trajectories)

**TODO(conv-point):** populate `ACTION_VOCAB` from a fresh B3 collection run. The frozen list locks the MLP's output indices for the entire training run.

In [ ]:
from collections import Counter

def collect_b3_action_vocab(num_episodes: int = 50) -> list[tuple]:
    """Run B3 for num_episodes and return the top-50 (kind, brain, subagent) tuples.

    TODO(conv-point): uncomment the body when B3CortexFixedRouter exists.
    """
    counter: Counter = Counter()
    # TODO(conv-point):
    # b3 = B3CortexFixedRouter(env=...)
    # for ep in range(num_episodes):
    #     trajectory = b3.run_episode(task="outbreak_easy", seed=ep)
    #     for router_step in trajectory.router_steps:
    #         action = router_step.routing_action
    #         key = (action.kind, getattr(action, "brain", None),
    #                getattr(action, "subagent", None))
    #         counter[key] += 1
    # The count is empty until Session 13 lands; callers must populate.
    return [k for k, _ in counter.most_common(ACTION_VOCAB_SIZE)]

ACTION_VOCAB: list[tuple] = collect_b3_action_vocab(num_episodes=50)
print(f"ACTION_VOCAB size = {len(ACTION_VOCAB)} (will be 50 at convergence point)")

## 7. Collect training data: B3 trajectories

**TODO(conv-point):** B3CortexFixedRouter runs ~50 episodes. Each `RouterStep` is a (state_features, action_idx, reward) tuple. The MLP is trained to predict good actions given state.

In [ ]:
TRAIN_EPISODES = 50

def collect_b3_trajectories(num_episodes: int = TRAIN_EPISODES) -> RolloutBuffer:
    """Run B3 deterministic-router rollouts and return a populated RolloutBuffer.

    Stores per-router-step tuples: (featurized_state, action_idx, reward, log_prob=None, done).
    log_prob is None because B3 is deterministic (no sampling).

    TODO(conv-point): uncomment the body when B3 ships.
    """
    buf = RolloutBuffer()
    # TODO(conv-point):
    # b3 = B3CortexFixedRouter(env=...)
    # for ep in range(num_episodes):
    #     trajectory = b3.run_episode(task="outbreak_easy", seed=ep)
    #     ep_id = f"b3:outbreak_easy:{ep}"
    #     for router_step in trajectory.router_steps:
    #         feat = featurize_metacognition_state(router_step.metacognition_state)
    #         action_key = (router_step.routing_action.kind,
    #                       getattr(router_step.routing_action, "brain", None),
    #                       getattr(router_step.routing_action, "subagent", None))
    #         action_idx = ACTION_VOCAB.index(action_key) if action_key in ACTION_VOCAB else -1
    #         buf.add_step(ep_id, TrajectoryStep(
    #             obs={"feat": feat.tolist()},
    #             action={"idx": action_idx},
    #             reward=0.0,  # backfilled per-tick by trainer
    #             log_prob=None,
    #             done=False,
    #         ))
    return buf

buf = collect_b3_trajectories()
print(f"Buffer size = {len(buf)} (will be {TRAIN_EPISODES} at conv-point)")

## 8. GRPO training loop

Group sampling -> relative advantage -> policy gradient. Reuses Phase-2 `RolloutBuffer` and `shape_reward`.

In [ ]:
TRAIN_STEPS = 300
GROUP_SIZE = 8
LR = 1e-3
TEMPERATURE = 1.0

optimizer = torch.optim.Adam(policy.parameters(), lr=LR)
loss_history: list[float] = []

def grpo_step(buffer: RolloutBuffer) -> float:
    """One GRPO update step.

    1. Sample GROUP_SIZE episodes.
    2. For each sampled (state, action_idx, reward) tuple, compute relative
       advantage = reward - mean(group_rewards).
    3. policy_loss = -mean(advantage * log_prob(action | state))
    4. step.
    """
    if len(buffer) < GROUP_SIZE:
        return 0.0
    ep_ids = buffer.sample_group(GROUP_SIZE)
    feats: list[torch.Tensor] = []
    action_idxs: list[int] = []
    rewards: list[float] = []
    for ep_id in ep_ids:
        for step in buffer.get_episode(ep_id):
            feats.append(torch.tensor(step.obs["feat"], dtype=torch.float32))
            action_idxs.append(step.action["idx"])
            rewards.append(step.reward)
    if not feats:
        return 0.0
    feat_t = torch.stack(feats)
    action_t = torch.tensor(action_idxs, dtype=torch.long)
    reward_t = torch.tensor(rewards, dtype=torch.float32)
    advantage = reward_t - reward_t.mean()
    logits = policy(feat_t) / TEMPERATURE
    log_prob = torch.distributions.Categorical(logits=logits).log_prob(action_t)
    loss = -(advantage * log_prob).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return float(loss.item())

# TODO(conv-point): replace the placeholder loop below with real per-step
# rollout (featurize -> policy.sample_action -> council.step -> reward shaping ->
# buffer.add_step -> grpo_step). Until then, this only exercises the loss math.
for step in range(min(TRAIN_STEPS, 1)):
    loss = grpo_step(buf)
    loss_history.append(loss)
    if step % 25 == 0:
        print(f"step={step} loss={loss:.4f} buffer_size={len(buf)}")

## 9. Save the trained MLP weights

Persists state_dict + config to HF Hub. Lightweight (a few hundred KB).

In [ ]:
from huggingface_hub import HfApi

HUB_REPO = "Angshuman28/crisisworld-cortex-router-mlp"

torch.save({
    "state_dict": policy.state_dict(),
    "feature_dim": FEATURE_DIM,
    "vocab_size": ACTION_VOCAB_SIZE,
    "hidden_dim": HIDDEN_DIM,
    "action_vocab": ACTION_VOCAB,
}, "/content/cortex_router_mlp.pt")

api = HfApi()
api.create_repo(HUB_REPO, exist_ok=True, repo_type="model", private=False, token=HF_TOKEN)
api.upload_file(
    path_or_fileobj="/content/cortex_router_mlp.pt",
    path_in_repo="cortex_router_mlp.pt",
    repo_id=HUB_REPO,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"Saved to https://huggingface.co/{HUB_REPO}")

## 10. Eval: B6 (trained MLP router) vs B3 (deterministic) on 3 tasks

**TODO(conv-point):** the comparison loop below requires both B3CortexFixedRouter (deterministic) and a way to swap the trainable MLP into Council.routing_policy. The eval is the headline result for the convergence point — its sign decides whether B6 ships or B3 ships per the Phase 5 hard-exit gate.

Decision rule (per spec): if reward over training steps is INCREASING, ship B6. Else ship B3.

In [ ]:
# TODO(conv-point):
# from cortex.routing_policy import DeterministicRouter, TrainableRouter
# from cortex.council import Council
# from baselines.cortex_fixed_router import B3CortexFixedRouter
# from CrisisWorldCortex.client import CrisisworldcortexEnv
#
# def make_env():
#     return CrisisworldcortexEnv(base_url="https://angshuman28-crisisworldcortex.hf.space")
#
# def run_b6_episode(task, seed):
#     trainable = TrainableRouter(policy=policy, action_vocab=ACTION_VOCAB,
#                                 featurize=featurize_metacognition_state)
#     council = Council(routing_policy=trainable, env=make_env())
#     return council.run_episode(task=task, seed=seed).total_reward
#
# def run_b3_episode(task, seed):
#     b3 = B3CortexFixedRouter(env=make_env())
#     return b3.run_episode(task=task, seed=seed).total_reward
#
# TASKS = ("outbreak_easy", "outbreak_medium", "outbreak_hard")
# b6_results = {t: run_b6_episode(t, 0) for t in TASKS}
# b3_results = {t: run_b3_episode(t, 0) for t in TASKS}
# print(f"B6 (trained MLP): {b6_results}")
# print(f"B3 (deterministic): {b3_results}")

print("Eval cell skeleton — uncomment when Cortex Session 13 ships")

## 11. Plot training loss

Phase 5 hard-exit gate watches this curve. If reward is increasing across training steps -> ship B6. Else -> ship B3.

In [ ]:
import matplotlib.pyplot as plt

if loss_history:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(loss_history)
    ax.set_xlabel("step")
    ax.set_ylabel("GRPO policy loss")
    ax.set_title("Cortex MLP router — training loss")
    plt.tight_layout()
    plt.show()
else:
    print("No loss history yet — uncomment the TODO blocks at conv-point")